# Evaluación formal del modelo principal — modelo perinatal NCHS (opción A, prenatal)

Etapas del framework: **E8** (modelado, validación y métricas), **E9** (sensibilidad y robustez) y puerta **P4** (clasificación). Evalúa el modelo principal elegido en la corrida preliminar (XGBoost, con los hiperparámetros ganadores de la búsqueda aleatoria) y la regresión logística como referencia. Versión en inglés: `05_main_evaluation.ipynb` (mismo código).

**Diseño.**
- **Validación interna:** validación cruzada estratificada repetida, 5 particiones × 3 repeticiones, sobre los años de desarrollo (2016–2021). En cada partición de entrenamiento el modelo se ajusta con el 80% y se **recalibra con Platt** en el 20% restante. Intervalos del 95% por bootstrap (200 réplicas). Los productos son unidades independientes: los archivos no identifican a la madre.
- **Hiperparámetros fijos:** se toman de la corrida preliminar. Como en esa búsqueda las 20 combinaciones de XGBoost quedaron a menos de 0,004 de AUROC entre sí, no se repite la búsqueda dentro de cada partición (la validación anidada completa del framework); el optimismo por esta simplificación es despreciable con estos tamaños.
- **Muestra:** todos los eventos y el 2% de los no eventos, con pesos, de modo que la calibración, el AUPRC y las matrices de confusión representen a la población completa.
- **Validación temporal (C4):** ajuste con 2016–2021 y una sola prueba en 2022–2023.
- **Sensibilidades (E9):** solo registros completos (D3), 28 semanas o más, solo muertes fetales, solo muertes fetales anteparto y solo muertes neonatales precoces.
- **Clasificación (P4):** C1 se cumple por diseño (toda la información es previa a cualquier evento); C2 se juzga con la AUROC de la población principal (0,75 o más, con límite inferior de 0,70 o más); C3 no aplica; C4 exige AUROC temporal de 0,70 o más y una pendiente de calibración cuyo IC 95% incluya 1.
- También se reportan la matriz de confusión y la exactitud (pedidas, no son métricas del framework) y la curva de decisión.

Salida: `data/outputs_05_main_evaluation.txt`. Tarda del orden de 1 hora.

In [1]:
import sys, os, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "perinatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_05_main_evaluation.txt"
SEED = 20261121
DEV_YEARS, TEST_YEARS = list(range(2016, 2022)), [2022, 2023]
FRACTION = float(os.environ.get("NCHS_FRACTION", 0.02))
QUICK = os.environ.get("NCHS_QUICK") == "1"
N_JOBS = int(os.environ.get("NCHS_JOBS", -1))
REPEATS, FOLDS, B = (1, 3, 30) if QUICK else (3, 5, 200)
PREDICTORS = ['age', 'race_black', 'race_hispanic', 'race_asian', 'race_other', 'educ_some_college',
              'educ_bachelor_plus', 'born_outside_us', 'wic', 'prior_living_1', 'prior_living_2plus',
              'prior_child_died', 'interval_lt18', 'interval_18_59', 'interval_60plus', 'prior_cesarean',
              'bmi', 'smoked_before_pregnancy', 'care_after_first_trimester', 'no_prenatal_care',
              'pregestational_diabetes', 'chronic_hypertension', 'infertility_treatment', 'multiple_pregnancy']
EXTRA = ["label_main", "label_fetal", "label_early_neonatal", "ga_weeks", "estofd", "n_unknown", "period"]

# Hyperparameters chosen by the preliminary run (notebook 04, random search, 2026-11-22). They are
# fixed here; the primary learner is XGBoost (selection rule) and logistic regression is the reference.
XGB_PARAMS = dict(max_depth=7, learning_rate=0.0105, min_child_weight=79.6815, subsample=0.9275,
                  colsample_bytree=0.6472, reg_lambda=5.5315, reg_alpha=0.0429, n_estimators=50 if QUICK else 695)
LR_PARAMS = dict(C=0.0002, penalty="l2", solver="liblinear", max_iter=500)

def make_model(name):
    if name == "XGB":
        return make_pipeline(SimpleImputer(strategy="median"),
                             XGBClassifier(tree_method="hist", n_jobs=N_JOBS, random_state=SEED, **XGB_PARAMS))
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(**LR_PARAMS))

lines = []
def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

## 1. Muestras de desarrollo y de prueba temporal

In [2]:
# Samples: every event and FRACTION of non-events, weight 1/FRACTION for non-events (the weighted
# sample stands for the full population). Development 2016-2021; temporal test 2022-2023.
def load(years, seed):
    rng = np.random.default_rng(seed); parts = []; n = e = 0
    for y in years:
        c = pd.read_parquet(CONTRACT_DIR / f"year_{y}.parquet", columns=PREDICTORS + EXTRA)
        n += len(c); e += int(c["label_main"].sum())
        parts.append(c[c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)])
    d = pd.concat(parts, ignore_index=True)
    if QUICK:
        d = d.sample(n=min(len(d), 40000), random_state=seed).reset_index(drop=True)
    d["w"] = np.where(d["label_main"], 1.0, 1.0 / FRACTION)
    return d, n, e

dev, n_dev, e_dev = load(DEV_YEARS, SEED)
test, n_test, e_test = load(TEST_YEARS, SEED + 1)
show("Data", pd.DataFrame({"products": [n_dev, n_test], "events": [e_dev, e_test],
                           "sample rows": [len(dev), len(test)]}, index=["development 2016-2021", "temporal test 2022-2023"]))


== Data
                         products  events  sample rows
development 2016-2021    22681046  149057       600442
temporal test 2022-2023   7282230   45117       189841


## 2. Métricas y validación interna recalibrada

In [3]:
# Metrics (weighted where the sampling matters; AUROC does not depend on it)
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
expit = lambda z: 1 / (1 + np.exp(-z))

def cal_slope_intercept(y, p, w):
    lp = logit(p)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y, sample_weight=w).coef_[0, 0]
    f = lambda a: np.sum(w * (y - expit(a + lp)))        # calibration-in-the-large (offset = lp)
    return float(slope), float(brentq(f, -10, 10))

def metrics(y, p, w):
    o = np.argsort(p); cw = np.cumsum(w[o]); m = {}
    for share in (0.05, 0.10):
        cut = p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
        m[f"events in top {int(share * 100)}% (%)"] = 100 * float(w[(p >= cut) & y].sum() / w[y].sum())
    slope, intercept = cal_slope_intercept(y, p, w)
    g = np.empty(len(p), int); g[o] = np.minimum((10 * cw / cw[-1]).astype(int), 9)   # weighted deciles
    ece = sum(abs(np.sum(w[g == k] * y[g == k]) - np.sum(w[g == k] * p[g == k])) for k in range(10)) / w.sum()
    m.update({"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p, sample_weight=w),
              "prevalence": float(np.sum(w * y) / w.sum()), "Brier": float(np.sum(w * (p - y) ** 2) / w.sum()),
              "calibration slope": slope, "calibration intercept": intercept,
              "observed/expected": float(np.sum(w * y) / np.sum(w * p)), "ECE": float(ece)})
    return m

def boot_ci(y, p, w, keys, seed):
    rng = np.random.default_rng(seed); pos, neg = np.where(y)[0], np.where(~y)[0]; res = []
    for _ in range(B):
        ix = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        res.append({k: v for k, v in metrics(y[ix], p[ix], w[ix]).items() if k in keys})
    r = pd.DataFrame(res)
    return r.quantile(0.025), r.quantile(0.975)

def fit_calibrated(name, X, y, w, seed):
    """Fit on 80% of the training part and Platt-recalibrate on the other 20% (inside training)."""
    X_f, X_c, y_f, y_c, w_f, w_c = train_test_split(X, y, w, test_size=0.2, stratify=y, random_state=seed)
    m = make_model(name); step = m.steps[-1][0]
    m.fit(X_f, y_f, **{f"{step}__sample_weight": w_f})
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(m.predict_proba(X_c)[:, 1]).reshape(-1, 1), y_c, sample_weight=w_c)
    return lambda Xn: platt.predict_proba(logit(m.predict_proba(Xn)[:, 1]).reshape(-1, 1))[:, 1]

# Repeated stratified cross-validation (products are independent units; no mother identifier):
# REPEATS x FOLDS, recalibration inside each training part, out-of-fold predictions pooled per repeat.
X_dev, y_dev, w_dev = dev[PREDICTORS], dev["label_main"].to_numpy(bool), dev["w"].to_numpy()
oof = {name: np.zeros((REPEATS, len(dev))) for name in ("XGB", "LR")}
for r in range(REPEATS):
    t = time.time()
    for tr, te in StratifiedKFold(FOLDS, shuffle=True, random_state=SEED + r).split(X_dev, y_dev):
        for name in oof:
            f = fit_calibrated(name, X_dev.iloc[tr], y_dev[tr], w_dev[tr], SEED + r)
            oof[name][r, te] = f(X_dev.iloc[te])
    print(f"repeat {r + 1} done in {(time.time() - t) / 60:.1f} min")

KEYS = ["AUROC", "AUPRC", "calibration slope", "calibration intercept", "observed/expected"]
rows = {}
for name in oof:
    per_rep = pd.DataFrame([metrics(y_dev, oof[name][r], w_dev) for r in range(REPEATS)])
    lo, hi = boot_ci(y_dev, oof[name][0], w_dev, KEYS, SEED)
    row = per_rep.mean().round(4).to_dict()
    row.update({f"{k} 2.5%": round(lo[k], 4) for k in KEYS}); row.update({f"{k} 97.5%": round(hi[k], 4) for k in KEYS})
    row["AUROC SD across repeats"] = round(per_rep["AUROC"].std(ddof=0), 4)
    rows[name] = row
main = pd.DataFrame(rows)
show(f"Internal validation: {REPEATS} x {FOLDS}-fold cross-validation, recalibrated (development 2016-2021)", main)

repeat 1 done in 1.5 min
repeat 2 done in 1.4 min
repeat 3 done in 1.5 min

== Internal validation: 3 x 5-fold cross-validation, recalibrated (development 2016-2021)
                                 XGB       LR
events in top 5% (%)         30.9340  26.7323
events in top 10% (%)        41.1762  38.4587
AUROC                         0.7389   0.7220
AUPRC                         0.0579   0.0351
prevalence                    0.0066   0.0066
Brier                         0.0063   0.0064
calibration slope             1.0280   0.9627
calibration intercept         0.0066   0.0005
observed/expected             1.0065   1.0004
ECE                           0.0002   0.0004
AUROC 2.5%                    0.7374   0.7206
AUPRC 2.5%                    0.0556   0.0341
calibration slope 2.5%        1.0229   0.9620
calibration intercept 2.5%    0.0037  -0.0028
observed/expected 2.5%        1.0036   0.9973
AUROC 97.5%                   0.7402   0.7235
AUPRC 97.5%                   0.0610   0.0366
calibr

## 3. Validación temporal (C4) y análisis de sensibilidad (E9)

In [4]:
# Temporal validation (criterion C4): fit on the whole development sample, recalibrate inside it,
# apply once to 2022-2023.
X_t, y_t, w_t = test[PREDICTORS], test["label_main"].to_numpy(bool), test["w"].to_numpy()
temporal = {}
for name in ("XGB", "LR"):
    f = fit_calibrated(name, X_dev, y_dev, w_dev, SEED)
    p_t = f(X_t)
    m = metrics(y_t, p_t, w_t); lo, hi = boot_ci(y_t, p_t, w_t, KEYS, SEED + 7)
    temporal[name] = {**{k: round(v, 4) for k, v in m.items()},
                      **{f"{k} 2.5%": round(lo[k], 4) for k in KEYS}, **{f"{k} 97.5%": round(hi[k], 4) for k in KEYS}}
    if name == "XGB":
        p_temporal_xgb = p_t
show("Temporal validation: development 2016-2021, single test 2022-2023", pd.DataFrame(temporal))

# Sensitivity analyses of the outcome card (E9), on the pooled out-of-fold predictions of XGBoost (repeat 1)
p0 = oof["XGB"][0]
complete = (dev["n_unknown"] == 0).to_numpy()
ge28 = (dev["ga_weeks"] >= 28).to_numpy()
is_fd = dev["label_fetal"].to_numpy(bool); is_en = dev["label_early_neonatal"].to_numpy(bool)
antepartum = dev["estofd"].astype("string").eq("N").fillna(False).to_numpy(bool)
no_event = ~y_dev
sens = {
    "all products (main)": (np.ones(len(dev), bool), y_dev),
    "complete records only (D3)": (complete, y_dev),
    "28 weeks or more": (ge28, y_dev),
    "fetal deaths vs no event": (is_fd | no_event, is_fd),
    "antepartum fetal deaths vs no event": (antepartum | no_event, antepartum),
    "early neonatal deaths vs no event": (is_en | no_event, is_en),
}
srows = {}
for k, (mask, yy) in sens.items():
    srows[k] = {"products (weighted)": int(round(w_dev[mask].sum())), "events": int(yy[mask].sum()),
                "AUROC": round(roc_auc_score(yy[mask], p0[mask]), 4)}
show("Sensitivity analyses (XGBoost, out-of-fold predictions)", pd.DataFrame(srows).T)


== Temporal validation: development 2016-2021, single test 2022-2023
                                 XGB       LR
events in top 5% (%)         29.5343  25.8816
events in top 10% (%)        40.1844  37.2631
AUROC                         0.7345   0.7164
AUPRC                         0.0493   0.0319
prevalence                    0.0062   0.0062
Brier                         0.0060   0.0061
calibration slope             0.9809   0.9812
calibration intercept        -0.1090  -0.1080
observed/expected             0.9000   0.8999
ECE                           0.0007   0.0008
AUROC 2.5%                    0.7319   0.7136
AUPRC 2.5%                    0.0462   0.0299
calibration slope 2.5%        0.9322   0.9804
calibration intercept 2.5%   -0.1180  -0.1160
observed/expected 2.5%        0.8922   0.8931
AUROC 97.5%                   0.7375   0.7192
AUPRC 97.5%                   0.0542   0.0342
calibration slope 97.5%       0.9819   0.9827
calibration intercept 97.5%  -0.0989  -0.1009
observed/e

## 4. Puerta P4, matriz de confusión y curva de decisión

In [5]:
# Gate G4 (classification). C1: with option A the information precedes every event (applies by design).
# C2: every event is dated after the prediction point, so C2 is judged on the AUROC of the main
# population (>= 0.75 with lower 95% limit >= 0.70). C3: not applicable. C4: temporal AUROC >= 0.70
# and calibration correctable by the intercept (95% CI of the slope includes 1).
x, tt = main["XGB"], temporal["XGB"]
c2 = (x["AUROC"] >= 0.75) and (x["AUROC 2.5%"] >= 0.70)
c4 = (tt["AUROC"] >= 0.70) and (tt["calibration slope 2.5%"] <= 1 <= tt["calibration slope 97.5%"])
top10 = x["events in top 10% (%)"] / 10
if x["AUROC"] >= 0.75 and c2 and c4:
    cls = "PREDICTOR"
elif x["AUROC"] >= 0.75 and top10 >= 2:
    cls = "EARLY IDENTIFIER"
else:
    cls = "NEITHER"
if tt["AUROC"] < 0.65:
    cls = "NEITHER"
show("Gate G4: classification of the primary model (XGBoost)", pd.Series({
    "C1 events already present": "met by design (antenatal information precedes every event)",
    "C2 outcome with time": f"{'met' if c2 else 'NOT met'} (AUROC {x['AUROC']:.3f}, 95% CI {x['AUROC 2.5%']:.3f}-{x['AUROC 97.5%']:.3f}; required >= 0.75 and >= 0.70)",
    "C3 detection bias": "not applicable (the outcome does not depend on a test)",
    "C4 stability over time": f"{'met' if c4 else 'NOT met'} (temporal AUROC {tt['AUROC']:.3f}; slope {tt['calibration slope']:.2f}, 95% CI {tt['calibration slope 2.5%']:.2f}-{tt['calibration slope 97.5%']:.2f})",
    "events in top 10% vs chance": f"{top10:.2f} times",
    "classification": cls}))

# Accuracy and confusion matrix of the primary model (requested; not a framework metric)
cm = []
for share in (0.10, 0.05):
    p = oof["XGB"][0]; o = np.argsort(p); cw = np.cumsum(w_dev[o])
    flag = p >= p[o][np.searchsorted(cw, (1 - share) * cw[-1])]
    tp, fn = w_dev[flag & y_dev].sum(), w_dev[~flag & y_dev].sum()
    fp, tn = w_dev[flag & ~y_dev].sum(), w_dev[~flag & ~y_dev].sum()
    cm.append({"flagged": f"top {int(share * 100)}%", "TP": round(tp), "FN": round(fn), "FP": round(fp), "TN": round(tn),
               "accuracy": round((tp + tn) / (tp + tn + fp + fn), 4), "sensitivity": round(tp / (tp + fn), 3),
               "specificity": round(tn / (tn + fp), 4), "PPV": round(tp / (tp + fp), 4), "NPV": round(tn / (tn + fn), 5)})
show("Confusion matrix of XGBoost (out-of-fold, weighted to the development population)", pd.DataFrame(cm).set_index("flagged"))

# Decision curve (net benefit) at a few risk thresholds
dc = []
for thr in (0.005, 0.01, 0.02, 0.05):
    p = oof["XGB"][0]; flag = p >= thr; n = w_dev.sum()
    nb = w_dev[flag & y_dev].sum() / n - w_dev[flag & ~y_dev].sum() / n * thr / (1 - thr)
    prev = w_dev[y_dev].sum() / n
    dc.append({"threshold": thr, "net benefit model": round(nb, 5), "net benefit treat all": round(prev - (1 - prev) * thr / (1 - thr), 5),
               "% flagged": round(100 * w_dev[flag].sum() / n, 2)})
show("Decision curve (XGBoost)", pd.DataFrame(dc).set_index("threshold"))

OUT_FILE.write_text("Main evaluation of the perinatal model (antenatal, option A)\n" + "\n".join(lines) + "\n", encoding="utf-8")


== Gate G4: classification of the primary model (XGBoost)
C1 events already present      met by design (antenatal information precedes ...
C2 outcome with time           NOT met (AUROC 0.739, 95% CI 0.737-0.740; requ...
C3 detection bias              not applicable (the outcome does not depend on...
C4 stability over time         NOT met (temporal AUROC 0.735; slope 0.98, 95%...
events in top 10% vs chance                                           4.12 times
classification                                                           NEITHER

== Confusion matrix of XGBoost (out-of-fold, weighted to the development population)
            TP      FN       FP        TN  accuracy  sensitivity  specificity     PPV      NPV
flagged                                                                                       
top 10%  61417   87640  2210450  20358800    0.8988        0.412       0.9021  0.0270  0.99571
top 5%   46087  102970  1089850  21479400    0.9475        0.309       0.9517  0.040

4377